<a href="https://colab.research.google.com/github/chomingi-25/2026_tues_bigdatacomputing_class/blob/main/%EB%8B%A4%EC%A4%91_%ED%8A%B9%EC%84%B1_%ED%9A%8C%EA%B7%80_%EB%AA%A8%EB%8D%B8_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8_%EA%B5%AC%EC%B6%95_%EB%B0%8F_Streamlit_%EC%9B%B9_%EC%84%9C%EB%B9%84%EC%8A%A4_%EB%B0%B0%ED%8F%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0단계
- 실습 환경 구축

In [ ]:
!pip install streamlit -q
!pip install pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 36.3 MB/s eta 0:00:00


## 1단계
- 데이터 준비 및 소규모 훈련 샘플링
- 파이프라인 기반 모델 3종 학습 및 저장

In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error

url="https://github.com/dongupak/DataML/raw/main/csv/life_expectancy.csv"
df=pd.read_csv(url)
df.columns = df.columns.str.strip()
df=df.dropna()
features=['Adult mortality', 'BMI', 'GDP', 'Alcohol']
target='Life expectancy'
X=df[features]
y=df[target]
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=100)
X_train_small=X_train.sample(n=50, random_state=100)
y_train_small=y_train.loc[X_train_small.index]
linear_model=Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])
poly_model=Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=3)),
    ('lr', LinearRegression())
])
ridge_model=Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=3)),
    ('ridge', Ridge(alpha=1.0))
])
linear_model.fit(X_train_small, y_train_small)
poly_model.fit(X_train_small, y_train_small)
ridge_model.fit(X_train_small, y_train_small)
joblib.dump(linear_model, 'linear_model.pkl')
joblib.dump(poly_model, 'poly_model.pkl')
joblib.dump(ridge_model, 'ridge.pkl')
print("모델 저장 완료")

모델 저장 완료


## 2단계
- app.py 실행

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
url="https://github.com/dongupak/DataML/raw/main/csv/life_expectancy.csv"
df=pd.read_csv(url)
df=df.dropna()
features=[
    'Adult mortality',
    'BMI',
    'GDP',
    'Alcohol'
]
target='Life expectancy'
X=df[features]
y=df[target]
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=100)
X_train_small=X_train.sample(n=50, random_state=100)
y_train_small=y_train.loc[X_train_small.index]
linear=joblib.load("linear_model.pkl")
poly=joblib.load("poly_model.pkl")
ridge=joblib.load("ridge.pkl")
models={
    "Linear": linear,
    "Poly": poly,
    'Ridge': ridge
}
result=[]
for name, model in models.items():
  train_pred=model.predict(X_train_small)
  test_pred=model.predict(X_test)
  train_r2=r2_score(y_train_small, train_pred)
  test_r2=r2_score(y_test, test_pred)
  train_mse=mean_squared_error(y_train_small, train_pred)
  test_mse=mean_squared_error(y_test, test_pred)
  if name=="Linear":
    complexity=len(features)
  else:
    complexity=model.named_steps['poly'].n_output_features_
  result.append([
      name,
      train_r2,
      test_r2,
      train_mse,
      test_mse,
      complexity
  ])
score_df=pd.DataFrame(result,
                      columns=[
                          "Model",
                          "Train R2",
                          "Test R2",
                          "Train MSE",
                          "Test MSE",
                          "Complexity"
                      ]
)
st.title("Life Expectancy Prediction")
st.header("모델 성능 비교")
st.dataframe(score_df)
fig, ax=plt.subplots()
plot_df=score_df.copy()
plot_df["Test R2 clipped"]=plot_df["Test R2"].clip(-1, 1)
ax.bar(plot_df["Model"], plot_df["Test R2 clipped"])
ax.set_title("Test R2 Comparison")
st.pyplot(fig)
st.sidebar.header("입력값")
adult=st.sidebar.slider(
    "Adult Mortality",
    float(X["Adult mortality"].min()),
    float(X["Adult mortality"].max()),
    float(X["Adult mortality"].mean())
)
bmi=st.sidebar.slider(
    "BMI",
    float(X["BMI"].min()),
    float(X["BMI"].max()),
    float(X["BMI"].mean())
)
gdp=st.sidebar.slider(
    "GDP",
    float(X["GDP"].min()),
    float(X["GDP"].max()),
    float(X["GDP"].mean())
)
alcohol=st.sidebar.slider(
    "Alcohol",
    float(X["Alcohol"].min()),
    float(X["Alcohol"].max()),
    float(X["Alcohol"].mean())
)
selected_model=st.selectbox(
    "모델 선택",
    ["Linear", "Poly", "Ridge"]
)
input_df=pd.DataFrame({
    'Adult mortality':[adult],
    'BMI':[bmi],
    'GDP':[gdp],
    'Alcohol':[alcohol]
})
prediction=models[selected_model].predict(input_df)
st.header("예측 결과")
st.markdown("# {:.2f} 세".format(prediction[0]))

Overwriting app.py


## 3단계
- ngrok 연결

In [ ]:
!pip install streamlit pyngrok -q
from pyngrok import ngrok
import os

ngrok.kill()
ngrok.set_auth_token("3Dw1lJLpfOQWowerEZaVV7EPgLo_511PeWA4FgnT4YWE4LU36")
os.system("streamlit run app.py --server.address 127.0.0.1 &")
public_url=ngrok.connect(8501)
print("아래 링크를 클릭하세요:\n{}".format(public_url))

아래 링크를 클릭하세요:
NgrokTunnel: "https://snack-headfirst-unsalted.ngrok-free.dev" -> "http://localhost:8501"
